# ARC Prize 2026 — ARC-AGI-3 submission (team just2gurls)

**Agent:** `dqn` · **click targets:** `objects` · **no-op pruning:** `False` · **max actions per game:** 1000
**Code:** [github.com/Manani10/RL-Assignment-2026](https://github.com/Manani10/RL-Assignment-2026), commit `97611b3`; this notebook is generated from it by `scripts/build_kaggle_notebook.py`.

The agent learns **online, inside each game**: no pretrained model, no internet, no game-specific rules.
At every step it sees the 64×64 frame, chooses one of 70 actions (6 simple actions + 64 click targets;
with `objects`, each click target is the centre of a detected single-colour object), and receives
+1 when a level is completed.

| Cell | What it does |
|---|---|
| 1 | Install the ARC-AGI-3 toolkit from the competition's offline wheels |
| 2 | Write our library `arcrl` to `/tmp/arcrl_src` (one cell per source file, unchanged from the repository) |
| 3 | Write `my_agent.py`, the bridge between the official framework and our agent |
| 4 | Self-test on a synthetic frame (normal runs only) |
| 4b | Placeholder `submission.parquet` (must exist in every run, including scoring) |
| 5 | Competition rerun only: register the agent and play every hidden game (log: `framework.log`) |


In [ ]:
# 1) Install the competition's offline wheels (internet is disabled during scoring)
!pip install --no-index --find-links /kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels arc-agi python-dotenv -q


In [ ]:
# 2) Our library `arcrl`: create the package folders, then one cell per module below
import os
for d in ['/tmp/arcrl_src/arcrl', '/tmp/arcrl_src/arcrl/agents', '/tmp/arcrl_src/arcrl/env']:
    os.makedirs(d, exist_ok=True)


In [ ]:
%%writefile /tmp/arcrl_src/arcrl/__init__.py
"""arcrl â€” RL agents for ARC-AGI-3 (COMS4061A/COMS7071A group project)."""

__version__ = "0.1.0"


In [ ]:
%%writefile /tmp/arcrl_src/arcrl/agents/__init__.py
"""Agent registry. Torch agents are imported lazily so the random baseline and
the evaluation code work on machines without PyTorch."""
from __future__ import annotations


def build_agent(name: str, n_actions: int, in_ch: int, seed: int):
    if name == "random":
        from .random_agent import RandomAgent
        return RandomAgent(seed=seed)
    if name == "dqn":
        from .dqn import DQNAgent
        return DQNAgent(n_actions, in_ch, seed)
    if name == "dqn_imp1":
        from .dqn import DQNAgent, DQNConfig
        return DQNAgent(n_actions, in_ch, seed, cfg=DQNConfig(novelty=True, novelty_beta=0.01))
    if name == "ppo":
        from .ppo import PPOAgent
        return PPOAgent(n_actions, in_ch, seed)
    raise KeyError(f"Unknown agent {name!r}")


In [ ]:
%%writefile /tmp/arcrl_src/arcrl/agents/base.py
"""Common agent interface so the evaluation protocol treats every agent the same."""
from __future__ import annotations

import numpy as np


class Agent:
    """
    Lifecycle used by eval/protocol.py:
        agent.begin_game(game_id)          # new game: decide what persists
        a = agent.act(obs, info)           # choose an action index
        agent.observe(obs, a, r, next_obs, done, info)   # learning hook
    Document in the report which parameters persist between actions,
    levels, games and evaluation runs (assignment Â§4.4).
    """

    name = "base"

    def begin_game(self, game_id: str) -> None:
        pass

    def act(self, obs: np.ndarray, info: dict) -> int:
        raise NotImplementedError

    def observe(self, obs, action, reward, next_obs, done, info) -> None:
        pass

    def state_dict(self) -> dict:
        return {}


def masked_choice(rng: np.random.Generator, mask: np.ndarray) -> int:
    idx = np.flatnonzero(mask)
    return int(rng.choice(idx))


In [ ]:
%%writefile /tmp/arcrl_src/arcrl/agents/dqn.py
"""DQN baseline (algorithm covered in the course) â€” OWNER: Member 2.

Baseline = Double DQN with target network, uniform replay, epsilon-greedy over
the *allowed* actions only (action mask from the adapter).

What is learned : Q(s, a) for every discrete action index.
When            : every `train_every` env steps once the buffer has `learn_start` items.
Effect          : act = argmax_a Q(s, a) over allowed a (with prob 1-eps).

Improvement slots (fill in, one file/flag per improvement so ablations are easy):
    IMP1  (Member 2) count-based novelty bonus on grid hashes (exploration)
    IMP2  (shared, built by Member 1) structured action space: object click
          targets + no-op pruning, switched on via AdapterConfig flags
"""
from __future__ import annotations

from dataclasses import dataclass
from math import sqrt

import numpy as np
import torch
import torch.nn.functional as F
from torch import nn

from ..env.encoding import grid_hash
from .base import Agent, masked_choice
from .nets import GridEncoder


@dataclass
class DQNConfig:
    lr: float = 2.5e-4
    gamma: float = 0.99
    buffer_size: int = 50_000
    batch_size: int = 32
    learn_start: int = 500
    train_every: int = 4
    target_update: int = 1_000
    eps_start: float = 1.0
    eps_end: float = 0.05
    eps_decay_steps: int = 20_000
    persist_across_games: bool = False   # False = re-initialise per game (pure online)
    novelty: bool = False               # IMP1: add count-based intrinsic reward during training
    novelty_beta: float = 0.01          # IMP1: scale of the intrinsic reward
    device: str = "cuda" if torch.cuda.is_available() else "cpu"


class QNet(nn.Module):
    def __init__(self, in_ch: int, n_actions: int):
        super().__init__()
        self.enc = GridEncoder(in_ch)
        self.head = nn.Linear(256, n_actions)

    def forward(self, x):
        return self.head(self.enc(x))


class Replay:
    """Stores colour-index grids (int8, k x 64 x 64) instead of one-hot to save RAM."""

    def __init__(self, cap: int, k: int, n_actions: int):
        self.cap, self.i, self.full = cap, 0, False
        self.s = np.zeros((cap, k, 64, 64), np.int8)
        self.s2 = np.zeros_like(self.s)
        self.a = np.zeros(cap, np.int64)
        self.r = np.zeros(cap, np.float32)
        self.d = np.zeros(cap, np.float32)
        self.m2 = np.zeros((cap, n_actions), bool)

    def add(self, s, a, r, s2, d, m2):
        j = self.i
        self.s[j], self.a[j], self.r[j], self.s2[j], self.d[j], self.m2[j] = s, a, r, s2, d, m2
        self.i = (j + 1) % self.cap
        self.full = self.full or self.i == 0

    def __len__(self):
        return self.cap if self.full else self.i

    def sample(self, n, rng):
        idx = rng.integers(0, len(self), n)
        return self.s[idx], self.a[idx], self.r[idx], self.s2[idx], self.d[idx], self.m2[idx]


def to_index(obs: np.ndarray) -> np.ndarray:
    """(16k,64,64) one-hot -> (k,64,64) int8 colour indices."""
    k = obs.shape[0] // 16
    return obs.reshape(k, 16, 64, 64).argmax(1).astype(np.int8)


def to_onehot_batch(idx: np.ndarray, device) -> torch.Tensor:
    """(B,k,64,64) int8 -> (B,16k,64,64) float tensor."""
    t = torch.as_tensor(idx, dtype=torch.long, device=device)
    oh = F.one_hot(t, 16).permute(0, 1, 4, 2, 3).float()  # B,k,16,64,64
    return oh.reshape(t.shape[0], -1, 64, 64)


class DQNAgent(Agent):
    name = "dqn"

    def __init__(self, n_actions: int, in_ch: int = 16, seed: int = 0, cfg: DQNConfig | None = None):
        self.cfg = cfg or DQNConfig()
        self.n_actions, self.in_ch = n_actions, in_ch
        self.rng = np.random.default_rng(seed)
        torch.manual_seed(seed)
        # Counts are maintained for one (game, seed) evaluation run.
        # The evaluator calls begin_game once per run, not once per episode.
        self._novelty_counts: dict[str, int] = {}
        self._build()

    def _build(self):
        c = self.cfg
        self.q = QNet(self.in_ch, self.n_actions).to(c.device)
        self.q_t = QNet(self.in_ch, self.n_actions).to(c.device)
        self.q_t.load_state_dict(self.q.state_dict())
        self.opt = torch.optim.Adam(self.q.parameters(), lr=c.lr)
        self.buf = Replay(c.buffer_size, self.in_ch // 16, self.n_actions)
        self.t = 0

    def begin_game(self, game_id):
        # A fresh agent/run gets fresh novelty counts. Ordinary environment
        # episode resets do not call begin_game, so counts persist across them.
        self._novelty_counts.clear()
        if not self.cfg.persist_across_games:
            self._build()

    def _novelty_bonus(self, grid: np.ndarray) -> float:
        """Count a next-observed grid and return its inverse-sqrt count bonus."""
        state_key = grid_hash(grid)
        count = self._novelty_counts.get(state_key, 0) + 1
        self._novelty_counts[state_key] = count
        return self.cfg.novelty_beta / sqrt(count)

    def epsilon(self):
        c = self.cfg
        frac = min(1.0, self.t / c.eps_decay_steps)
        return c.eps_start + frac * (c.eps_end - c.eps_start)

    def act(self, obs, info):
        mask = info["action_mask"]
        if self.rng.random() < self.epsilon():
            return masked_choice(self.rng, mask)
        with torch.no_grad():
            x = torch.as_tensor(obs[None], device=self.cfg.device)
            q = self.q(x)[0].cpu().numpy()
        q[~mask] = -np.inf
        return int(q.argmax())

    def observe(self, obs, action, reward, next_obs, done, info):
        c = self.cfg
        # info describes next_obs. The shaped reward is stored only in replay;
        # the adapter reward and official ARC score are not modified.
        train_reward = float(reward)
        if c.novelty:
            train_reward += self._novelty_bonus(info["grid"])
        self.buf.add(
            to_index(obs), action, train_reward, to_index(next_obs),
            float(done), info["action_mask"]
        )
        self.t += 1
        if len(self.buf) >= c.learn_start and self.t % c.train_every == 0:
            self._learn()
        if self.t % c.target_update == 0:
            self.q_t.load_state_dict(self.q.state_dict())

    def _learn(self):
        c = self.cfg
        s, a, r, s2, d, m2 = self.buf.sample(c.batch_size, self.rng)
        s, s2 = to_onehot_batch(s, c.device), to_onehot_batch(s2, c.device)
        a = torch.as_tensor(a, device=c.device)
        r = torch.as_tensor(r, device=c.device)
        d = torch.as_tensor(d, device=c.device)
        m2 = torch.as_tensor(m2, device=c.device)
        q = self.q(s).gather(1, a[:, None]).squeeze(1)
        with torch.no_grad():  # Double DQN target with masked argmax
            a2 = self.q(s2).masked_fill(~m2, -1e9).argmax(1, keepdim=True)
            target = r + c.gamma * (1 - d) * self.q_t(s2).gather(1, a2).squeeze(1)
        loss = F.smooth_l1_loss(q, target)
        self.opt.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(self.q.parameters(), 10.0)
        self.opt.step()
        self.last_loss = float(loss)


In [ ]:
%%writefile /tmp/arcrl_src/arcrl/agents/nets.py
"""Shared CNN encoder. Use the SAME encoder for DQN and PPO so that differences
in results come from the learning algorithm, not the architecture."""
from __future__ import annotations

import torch
from torch import nn


class GridEncoder(nn.Module):
    """(B, C, 64, 64) one-hot -> (B, feat_dim)."""

    def __init__(self, in_ch: int, feat_dim: int = 256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, 32, 3, stride=2, padding=1), nn.ReLU(),   # 32x32
            nn.Conv2d(32, 64, 3, stride=2, padding=1), nn.ReLU(),      # 16x16
            nn.Conv2d(64, 64, 3, stride=2, padding=1), nn.ReLU(),      # 8x8
            nn.Flatten(),
            nn.Linear(64 * 8 * 8, feat_dim), nn.ReLU(),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


def masked_logits(logits: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
    return logits.masked_fill(~mask, -1e9)


In [ ]:
%%writefile /tmp/arcrl_src/arcrl/agents/random_agent.py
"""Sanity-check reference policy (required by the assignment, Â§3.2).

Uniform over currently-allowed actions. Not a learning baseline.
"""
from __future__ import annotations

import numpy as np

from .base import Agent, masked_choice


class RandomAgent(Agent):
    name = "random"

    def __init__(self, seed: int = 0, **_):
        self.rng = np.random.default_rng(seed)

    def act(self, obs, info):
        return masked_choice(self.rng, info["action_mask"])


In [ ]:
%%writefile /tmp/arcrl_src/arcrl/env/__init__.py
"""(package marker)"""


In [ ]:
%%writefile /tmp/arcrl_src/arcrl/env/actions.py
"""Discrete action space shared by the local adapter AND the Kaggle bridge.

Index layout (fixed size, so DQN/PPO networks never change shape):
    0..5            ACTION1, ACTION2, ACTION3, ACTION4, ACTION5, ACTION7
    6..6+N-1        ACTION6 clicks, N = (64 / click_cell)^2  (default 64)

Baseline   click_mode="grid":    click slot i = centre of cell i of a fixed grid.
IMPROVEMENT (shared, Member 1) â€” two independent flags:
    click_mode="objects": click slot i = a pixel inside the i-th detected object
        (4-connected region of one colour; background = most common colour and
        very large regions are excluded). Unused slots are masked out.
    prune_noops=True: remember, per (frame hash, action index), whether the
        action changed the frame; mask actions known to do nothing in the
        current frame. Never masks every action. Memory lasts for one game.

Both use only the public frame â€” no hidden game state.
"""
from __future__ import annotations

from dataclasses import dataclass

import numpy as np

from .encoding import grid_hash

SIMPLE_ACTIONS = ["ACTION1", "ACTION2", "ACTION3", "ACTION4", "ACTION5", "ACTION7"]
N_SIMPLE = len(SIMPLE_ACTIONS)


@dataclass
class ActionConfig:
    click_cell: int = 8                 # grid mode: 8x8 px cells -> 64 slots (also sets N)
    click_mode: str = "grid"            # "grid" | "objects"
    prune_noops: bool = False
    max_object_frac: float = 0.25       # objects larger than this fraction of the screen are ignored


def find_objects(grid: np.ndarray, max_frac: float = 0.25) -> list[tuple[int, int, int, int]]:
    """Connected single-colour regions (4-connectivity), background excluded.

    Returns [(size, y, x, colour)] where (y, x) is the member pixel closest to the
    region's centroid, sorted by (size, y, x) so the order is deterministic.
    """
    h, w = grid.shape
    colours, counts = np.unique(grid, return_counts=True)
    background = colours[counts.argmax()]
    seen = np.zeros_like(grid, dtype=bool)
    limit = max_frac * h * w
    objs = []
    for sy in range(h):
        for sx in range(w):
            if seen[sy, sx] or grid[sy, sx] == background:
                continue
            c = grid[sy, sx]
            stack, pix = [(sy, sx)], []
            seen[sy, sx] = True
            while stack:
                y, x = stack.pop()
                pix.append((y, x))
                for ny, nx in ((y - 1, x), (y + 1, x), (y, x - 1), (y, x + 1)):
                    if 0 <= ny < h and 0 <= nx < w and not seen[ny, nx] and grid[ny, nx] == c:
                        seen[ny, nx] = True
                        stack.append((ny, nx))
            if len(pix) > limit:
                continue
            p = np.asarray(pix)
            cy, cx = p.mean(0)
            k = int(((p[:, 0] - cy) ** 2 + (p[:, 1] - cx) ** 2).argmin())
            objs.append((len(pix), int(p[k, 0]), int(p[k, 1]), int(c)))
    objs.sort()
    return objs


class ActionSpace:
    def __init__(self, cfg: ActionConfig | None = None):
        self.cfg = cfg or ActionConfig()
        cell = self.cfg.click_cell
        n = 64 // cell
        self.grid_targets = [(y * cell + cell // 2, x * cell + cell // 2) for y in range(n) for x in range(n)]
        self.n_clicks = len(self.grid_targets)
        self.n = N_SIMPLE + self.n_clicks
        self.targets = list(self.grid_targets)
        self.noop: set[tuple[str, int]] = set()
        self._hash = None

    # -- per-game memory -------------------------------------------------
    def reset_memory(self):
        self.noop.clear()

    # -- per-step update -------------------------------------------------
    def update(self, grid: np.ndarray):
        """Call with every new frame before mask()/decode()."""
        self._hash = grid_hash(grid) if self.cfg.prune_noops else None
        if self.cfg.click_mode == "objects":
            objs = find_objects(grid, self.cfg.max_object_frac)[: self.n_clicks]
            self.targets = [(y, x) for _, y, x, _ in objs]
        else:
            self.targets = self.grid_targets

    def mask(self, allowed_names: set[str]) -> np.ndarray:
        m = np.zeros(self.n, dtype=bool)
        for i, name in enumerate(SIMPLE_ACTIONS):
            m[i] = name in allowed_names
        if "ACTION6" in allowed_names:
            m[N_SIMPLE:N_SIMPLE + len(self.targets)] = True
        if not m.any():
            m[:] = True  # never hand the agent an empty mask
        if self.cfg.prune_noops and self._hash is not None:
            pruned = m.copy()
            for i in np.flatnonzero(m):
                if (self._hash, int(i)) in self.noop:
                    pruned[i] = False
            if pruned.any():
                m = pruned
        return m

    def decode(self, idx: int) -> tuple[str, dict]:
        """Index -> (action name, data)."""
        if idx < N_SIMPLE:
            return SIMPLE_ACTIONS[idx], {}
        j = idx - N_SIMPLE
        y, x = self.targets[j] if j < len(self.targets) else self.grid_targets[j]
        return "ACTION6", {"x": int(x), "y": int(y)}

    def record(self, idx: int, changed: bool):
        """After a step: note whether action `idx` changed the frame it was taken in."""
        if self.cfg.prune_noops and self._hash is not None and not changed:
            self.noop.add((self._hash, int(idx)))


In [ ]:
%%writefile /tmp/arcrl_src/arcrl/env/encoding.py
"""Observation encoding shared by every agent.

ARC-AGI-3 frames are 64x64 grids with integer colours 0..15. A single env step
can return several frames (an animation); we keep the LAST one as the state and
optionally stack the previous K observations as a short history.

Keep this file the single source of truth so DQN and PPO see identical inputs
(required for a fair comparison).
"""
from __future__ import annotations

import hashlib
from collections import deque

import numpy as np

GRID = 64
N_COLOURS = 16


def last_grid(frame) -> np.ndarray:
    """Return the final 64x64 grid from whatever the SDK gives us.

    The SDK's `frame` field may be a 2-D grid or a list of 2-D grids
    (animation layers). We always return an int8 array of shape (64, 64).
    """
    arr = np.asarray(frame)
    if arr.ndim == 3:
        arr = arr[-1]
    if arr.ndim != 2:
        raise ValueError(f"Unexpected frame shape {arr.shape}")
    return arr.astype(np.int8)


def one_hot(grid: np.ndarray) -> np.ndarray:
    """(64,64) ints -> (16,64,64) float32 one-hot planes."""
    planes = np.zeros((N_COLOURS, *grid.shape), dtype=np.float32)
    idx = np.clip(grid, 0, N_COLOURS - 1).astype(np.int64)
    np.put_along_axis(planes, idx[None], 1.0, axis=0)
    return planes


def grid_hash(grid: np.ndarray) -> str:
    """Stable hash of a grid, used for novelty counts / state graphs."""
    return hashlib.blake2b(np.ascontiguousarray(grid).tobytes(), digest_size=8).hexdigest()


class FrameStack:
    """Keeps the last `k` one-hot grids -> tensor of shape (16*k, 64, 64)."""

    def __init__(self, k: int = 1):
        self.k = k
        self.buf: deque[np.ndarray] = deque(maxlen=k)

    def reset(self, grid: np.ndarray) -> np.ndarray:
        self.buf.clear()
        oh = one_hot(grid)
        for _ in range(self.k):
            self.buf.append(oh)
        return self.get()

    def push(self, grid: np.ndarray) -> np.ndarray:
        self.buf.append(one_hot(grid))
        return self.get()

    def get(self) -> np.ndarray:
        return np.concatenate(list(self.buf), axis=0)


## 3) Bridge: the official framework calls `choose_action`; we translate it to our agent's `act` / `observe`

In [ ]:
%%writefile /tmp/my_agent.py
"""Bridge between the official ARC-AGI-3 agent framework and our agents (Member 1).

The framework calls `choose_action(frames, latest_frame)` once per step and
`is_done(...)` to decide when to stop. We translate each call into our agent
API (`act` / `observe`, see src/arcrl/agents/base.py), so the SAME agent code
runs in our local experiments (scripts/evaluate.py) and on Kaggle.

Per step: encode the latest frame, update the action space (grid or object
click targets, src/arcrl/env/actions.py), give the agent the reward for the
previous action (+1 per newly completed level), and return its next action.
Learning is online only: the agent starts from scratch in every game, and the
internet is disabled during Kaggle evaluation.

Settings come from environment variables; scripts/build_kaggle_notebook.py
bakes the submitted agent's values in as the defaults.
"""
from __future__ import annotations

import os

import numpy as np
from arcengine import FrameData, GameAction, GameState  # noqa: F401

from agents.agent import Agent  # framework base class (from the starter's vendor/)

import sys
sys.path.insert(0, "/tmp/arcrl_src")  # our library, written by the cells above
from arcrl.agents import build_agent
from arcrl.env.actions import ActionConfig, ActionSpace
from arcrl.env.encoding import FrameStack, last_grid

AGENT_NAME = os.environ.get("ARCRL_AGENT", "dqn")   # e.g. dqn, ppo, dqn_imp1
CLICK_MODE = os.environ.get("ARCRL_CLICK_MODE", "objects")            # "grid" | "objects"
PRUNE_NOOPS = os.environ.get("ARCRL_PRUNE_NOOPS", "0") == "1"
CLICK_CELL = 8
FRAME_STACK = 1
MAX_ACTIONS = int(os.environ.get("ARCRL_MAX_ACTIONS", "1000"))


def _state(frame) -> str:
    return getattr(frame.state, "name", str(frame.state))


def _names(actions) -> set[str]:
    out = set()
    for a in actions or []:
        if isinstance(a, int):
            out.add(f"ACTION{a}")
        else:
            out.add(getattr(a, "name", str(a)))
    return out


class MyAgent(Agent):
    # The framework's main loop stops at self.MAX_ACTIONS (default 80) â€” use ours instead.
    MAX_ACTIONS = MAX_ACTIONS

    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        # same action space as local experiments (env/actions.py)
        self.actions = ActionSpace(ActionConfig(click_cell=CLICK_CELL, click_mode=CLICK_MODE,
                                                prune_noops=PRUNE_NOOPS))
        self.inner = build_agent(AGENT_NAME, self.actions.n, 16 * FRAME_STACK, seed=0)
        self.inner.begin_game(getattr(self, "game_id", "kaggle"))
        self.stack = FrameStack(FRAME_STACK)
        self.prev = None          # (obs, action_idx, grid)
        self.prev_levels = 0
        self.n = 0

    def is_done(self, frames, latest_frame) -> bool:
        return _state(latest_frame) == "WIN" or self.n >= MAX_ACTIONS

    def choose_action(self, frames, latest_frame):
        if _state(latest_frame) in ("NOT_PLAYED", "GAME_OVER"):
            self.prev = None
            self.n += 1
            return GameAction.RESET

        grid = last_grid(latest_frame.frame)
        if self.prev is not None:  # must run BEFORE update(): uses the previous frame's hash
            self.actions.record(self.prev[1], changed=not np.array_equal(grid, self.prev[2]))
        self.actions.update(grid)
        obs = self.stack.push(grid) if self.prev else self.stack.reset(grid)
        info = {"action_mask": self.actions.mask(_names(getattr(latest_frame, "available_actions", None)))}
        lv = int(latest_frame.levels_completed)
        if self.prev is not None:
            r = float(max(0, lv - self.prev_levels))
            self.inner.observe(self.prev[0], self.prev[1], r, obs, False, info)
        self.prev_levels = lv

        idx = self.inner.act(obs, info)
        self.prev = (obs, idx, grid)
        self.n += 1
        name, data = self.actions.decode(idx)
        action = getattr(GameAction, name)
        if data:
            action.set_data(data)
        return action


In [ ]:
# 4) Self-test (normal runs only): the library imports, the action space finds targets,
#    and the agent returns a legal action. Catches packaging errors before Kaggle scoring does.
import os, sys
import numpy as np

if not os.getenv("KAGGLE_IS_COMPETITION_RERUN"):
    sys.path.insert(0, "/tmp/arcrl_src")
    from arcrl.agents import build_agent
    from arcrl.env.actions import ActionConfig, ActionSpace
    from arcrl.env.encoding import FrameStack

    grid = np.zeros((64, 64), dtype=np.int64)
    grid[10:14, 20:24] = 3            # two small "objects" on a background
    grid[40:42, 50:56] = 7
    space = ActionSpace(ActionConfig(click_mode="objects", prune_noops=False))
    space.update(grid)
    mask = space.mask({"ACTION1", "ACTION2", "ACTION6"})
    agent = build_agent("dqn", space.n, 16, seed=0)
    agent.begin_game("self-test")
    a = agent.act(FrameStack(1).reset(grid), {"action_mask": mask})
    assert mask[a], "agent chose a masked-out action"
    print(f"self-test OK: agent={agent.__class__.__name__}, {int(mask.sum())} legal actions, chose {space.decode(a)}")


In [ ]:
# 4b) Placeholder submission.parquet, written BEFORE the agent plays and in every mode
#     (the official starter does the same: without this file the competition rerun ends in "Kaggle Error").
#     During scoring the gateway records the real results; this file only has to exist.
import pandas as pd
pd.DataFrame({"row_id": pd.Series(dtype=str), "game_id": pd.Series(dtype=str),
              "end_of_game": pd.Series(dtype=bool), "score": pd.Series(dtype=float)}) \
    .to_parquet("/kaggle/working/submission.parquet", index=False)
print("Placeholder submission.parquet written (zero rows).")


In [ ]:
# 5) Competition rerun only: plug our agent into the official framework and play every game.
#    The scoring gateway records every action; submission.parquet (cell 4b) must exist when the notebook ends.
import os, shutil, subprocess, pathlib

if os.getenv("KAGGLE_IS_COMPETITION_RERUN"):
    subprocess.run(["curl", "-s", "--retry", "60", "--retry-delay", "5", "--retry-connrefused",
                    "http://gateway:8001/api/games"], check=False)
    src = "/kaggle/input/competitions/arc-prize-2026-arc-agi-3/ARC-AGI-3-Agents"
    dst = "/kaggle/working/ARC-AGI-3-Agents"
    shutil.copytree(src, dst, dirs_exist_ok=True)
    shutil.copy("/tmp/my_agent.py", f"{dst}/agents/templates/my_agent.py")
    pathlib.Path(f"{dst}/agents/__init__.py").write_text(
        "from typing import Type\n"
        "from dotenv import load_dotenv\n"
        "from .agent import Agent, Playback\n"
        "from .swarm import Swarm\n"
        "from .templates.random_agent import Random\n"
        "from .templates.my_agent import MyAgent\n"
        "load_dotenv()\n"
        "AVAILABLE_AGENTS: dict[str, Type[Agent]] = {'random': Random, 'myagent': MyAgent}\n"
        "__all__ = ['Agent', 'Playback', 'Swarm', 'Random', 'MyAgent', 'AVAILABLE_AGENTS']\n"
    )
    pathlib.Path(f"{dst}/.env").write_text(
        "HOST=gateway\nPORT=8001\nOPERATION_MODE=online\n"
        "RECORDINGS_DIR=/kaggle/working/server_recording\n"
    )
    env = {**os.environ, "MPLBACKEND": "agg"}
    with open("/kaggle/working/framework.log", "w") as log:   # kept as notebook output for debugging
        r = subprocess.run(["python", "main.py", "--agent", "myagent"], cwd=dst, env=env,
                           stdout=log, stderr=subprocess.STDOUT)
    print("framework exit code:", r.returncode)  # never fail the notebook here: the gateway has the scores
